In [10]:
import csv

# Define input and output file paths
input_csv = 'indexing/SNP_haploregannotatr_ATAC_tss_SNPFunction_spliceai20240916.csv'  # Replace with your CSV file name
output_vcf = 'indexing/SNP_spliceai_20240916_v2.vcf'  # The output VCF file


In [11]:
# Contig lengths for hg38 (GRCh38)
contig_lengths = {
    '1': '248956422',
    '2': '242193529',
    '3': '198295559',
    '4': '190214555',
    '5': '181538259',
    '6': '170805979',
    '7': '159345973',
    '8': '145138636',
    '9': '138394717',
    '10': '133797422',
    '11': '135086622',
    '12': '133275309',
    '13': '114364328',
    '14': '107043718',
    '15': '101991189',
    '16': '90338345',
    '17': '83257441',
    '18': '80373285',
    '19': '58617616',
    '20': '64444167',
    '21': '46709983',
    '22': '50818468',
    'X': '156040895',
    'Y': '57227415',
    'MT': '16569'
}

# Function to convert chromosome labels to sortable keys
def chrom_to_key(chrom):
    # Remove 'chr' prefix if present
    chrom = chrom.replace('chr', '').strip()
    # Map numeric chromosomes to integers, others remain as strings
    if chrom.isdigit():
        return (0, int(chrom))  # Numeric chromosomes
    elif chrom == 'X':
        return (1, 23)
    elif chrom == 'Y':
        return (1, 24)
    elif chrom == 'MT' or chrom == 'M':
        return (1, 25)
    else:
        return (2, chrom)  # Other contigs

# Read variants from the CSV file and store them in a list
variants = []

with open(input_csv, 'r') as csv_file:
    csv_reader = csv.DictReader(csv_file)
    for row in csv_reader:
        # Extract and clean fields
        chrom = row['chr_number'].strip().replace('chr', '')  # Remove 'chr' prefix if present
        pos = row['pos_hg38'].strip()
        var_id = row['rsID'].strip()
        ref = row['a1'].strip()
        alt = row['a2'].strip()

        # Skip the row if any essential field is missing
        if not all([chrom, pos, ref, alt]):
            continue  # Skip this row

        # Append the variant as a dictionary
        variant = {
            'chrom': chrom,
            'pos': int(pos),
            'id': var_id if var_id else '.',
            'ref': ref,
            'alt': alt,
            'qual': '.',
            'filter': '.',
            'info': '.'
        }
        variants.append(variant)

# Sort the variants by chromosome and position
#variants.sort(key=lambda x: (chrom_to_key(x['chrom']), x['pos']))

# Open the output VCF file for writing
with open(output_vcf, 'w') as vcf_file:
    # Write the VCF header
    vcf_file.write('##fileformat=VCFv4.2\n')
    vcf_file.write('##fileDate=20231005\n')  # Update with the current date
    vcf_file.write('##reference=GRCh38/hg38\n')
    
    # Write contig lines
    for chrom, length in contig_lengths.items():
        vcf_file.write(f'##contig=<ID={chrom},length={length}>\n')
    
    # Write the column headers
    vcf_file.write('#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n')
    
    # Write the sorted variants to the VCF file
    for variant in variants:
        vcf_line = f"{variant['chrom']}\t{variant['pos']}\t{variant['id']}\t{variant['ref']}\t{variant['alt']}\t{variant['qual']}\t{variant['filter']}\t{variant['info']}\n"
        vcf_file.write(vcf_line)

conda activate spliceai

spliceai -I indexing/SNP_spliceai_20240916_v2.vcf -O spliceai_mpra_output_20240916_v2.vcf -R '/media/zihengc/T7/genome/hg38.fa' -A grch38

